# Constraint-preserving QAOA for QmRMR feature selection

This notebook is a self-contained demonstration of fixed-cardinality QmRMR feature selection. It introduces the mathematical model, constructs a number-preserving QAOA circuit with pyQPanda3, compares the variational result with exact enumeration and greedy forward selection, and records state-vector diagnostics.

## 1. Model and validation reference

Let r be a positive relevance vector and Q be a symmetric redundancy matrix. Selecting exactly k features is represented by a binary vector x. The canonical minimisation model is

$$
f(x)=x^{\mathsf T}Qx-r^{\mathsf T}x,
\qquad x_i\in\{0,1\},
\qquad \sum_{i=1}^{n}x_i=k.
$$

For small instances, the exact reference enumerates all feasible states. A greedy selector is included as a fast classical reference. Both references evaluate the same objective as the quantum circuit.

## 2. Ising encoding and constraint-preserving ansatz

The binary-to-Ising substitution is

$$
x_i=\frac{1-Z_i}{2},
\qquad
H_C=c+\sum_{i=1}^{n}h_iZ_i+\sum_{1\leq i\lt j\leq n}J_{ij}Z_iZ_j.
$$

The circuit begins in the fixed-weight Dicke state

$$
\lvert D_k^n\rangle=
\binom{n}{k}^{-1/2}
\sum_{\lVert x\rVert_1=k}\lvert x\rangle,
$$

and alternates a diagonal cost phase with the number-preserving XY mixer

$$
H_M=\frac{1}{2}\sum_{(i,j)\in E}
\left(X_iX_j+Y_iY_j\right).
$$

Every exchange term commutes with the Hamming-weight operator. Consequently, an ideal state-vector execution stays in the sector with exactly k selected features. A depth-p schedule contains 2p angles.

In [ ]:
from pathlib import Path
import sys
import numpy as np

PROJECT_ROOT = next(
    (candidate for candidate in (Path.cwd(), *Path.cwd().parents)
     if (candidate / 'pyqpanda_alg').is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError('Run this notebook from the project directory.')
sys.path.insert(0, str(PROJECT_ROOT))

from pyqpanda3.core import CPUQVM, QProg
from pyqpanda_alg.QmRMR.constrained import (
    ConstrainedQAOAQmRMR,
    ExchangeAnsatzSelector,
    QmRMRFeatureSelection,
    QmRMRProblem,
    greedy_forward_selection,
)
from pyqpanda_alg.QmRMR.constrained.circuits import build_dicke_state, parity_xy_mixer

In [ ]:
seed = 42
n_features = 6
select_num = 3
rng = np.random.default_rng(seed)
relevance = rng.random(n_features)
quadratic = rng.random((n_features, n_features))
quadratic = 0.5 * (quadratic + quadratic.T)

problem = QmRMRProblem.from_mrmr(
    quadratic,
    relevance,
    select_num,
    convention='canonical_mrmr',
)
exact = problem.exact_solution()
greedy = greedy_forward_selection(problem)

print('objective:', problem.objective_label)
print('exact state(s):', [problem.state_key(state) for state in exact.optimal_states])
print('exact value:', exact.optimal_value)
print('greedy state:', problem.state_key(greedy.state), 'value:', greedy.value)


## 3. High-level API

QmRMRFeatureSelection exposes the model and circuit choices through ordinary Python arguments. The constrained QAOA option uses the Dicke-plus-XY construction. The result object combines the selected state, expectation value, exact gap, feasibility, probability of an optimum, and circuit resources.

In [ ]:
selector = QmRMRFeatureSelection(
    quadratic,
    relevance,
    select_num,
    ansatz='constrained_qaoa',
    objective_convention='canonical_mrmr',
    layers=3,
    initializer='amplitude',
    optimization_objective='expectation',
    optimizer='slsqp',
    maxiter=20,
    seed=142,
    restarts=2,
)
qaoa_result = selector.optimize()
print(qaoa_result.summary())


In [ ]:
stages = selector.progressive_optimize(maxiter=12, seed=242, restarts=2)
for stage in stages:
    print({
        'layers': stage.metadata['layers'],
        'selected_state': stage.selected_key,
        'selected_gap': stage.optimum_gap,
        'expected_gap': stage.expected_gap,
        'feasible_probability': stage.feasible_probability,
        'optimum_probability': stage.optimum_probability,
        'warm_start_used': stage.metadata['warm_start_used'],
    })


## 4. Feasibility check in the CPU state-vector backend

The following smoke test prepares a Dicke state and applies an XY mixer without measurement. The probability dictionary is inspected directly; no sampling noise is introduced.

In [ ]:
qubits = list(range(n_features))
program = QProg(n_features)
program.append(build_dicke_state(qubits, select_num, mode='amplitude'))
program.append(parity_xy_mixer(qubits, beta=0.37, ring=True))
machine = CPUQVM()
machine.run(program, 1)
probabilities = machine.result().get_prob_dict(program.qubits())
feasible_mass = sum(
    probability
    for raw_key, probability in probabilities.items()
    if raw_key[::-1].count('1') == select_num
)
print('feasible probability:', feasible_mass)
assert abs(feasible_mass - 1.0) < 1e-10


## 5. Classical and exchange-ansatz comparison

The exchange ansatz is a second circuit construction exposed by the API. It preserves the fixed-weight sector and provides a compact circuit-level reference for the constrained QAOA result. All values below use feature-index order and the same exact objective.

In [ ]:
exchange = ExchangeAnsatzSelector(problem, bit_order='qubit').optimize(
    optimizer='slsqp',
    maxiter=20,
    seed=43,
    restarts=2,
)

comparison = {
    'exact': {
        'state': [problem.state_key(state) for state in exact.optimal_states],
        'value': exact.optimal_value,
    },
    'greedy': {
        'state': problem.state_key(greedy.state),
        'value': greedy.value,
    },
    'exchange': exchange.summary(),
    'constrained_qaoa': stages[-1].summary(),
}
comparison


In [ ]:
assert all(sum(state) == select_num for state in exact.optimal_states)
assert exchange.feasible_probability > 1.0 - 1e-10
assert stages[-1].feasible_probability > 1.0 - 1e-10
print('all fixed-cardinality checks passed')


## 6. Optional CVaR objective

Expectation minimisation uses the complete state-vector distribution. A lower-tail CVaR objective focuses the optimisation on the best objective values. For a minimisation problem,

$$
\operatorname{CVaR}_{\alpha}(f)=
\frac{1}{\alpha}\int_{0}^{\alpha}f_{\uparrow}(q)\,dq,
\qquad 0<\alpha\leq 1.
$$

The API reports both the value optimised by CVaR and the full expectation so that concentration and average quality can be inspected separately.

In [ ]:
cvar_selector = QmRMRFeatureSelection(
    quadratic,
    relevance,
    select_num,
    ansatz='constrained_qaoa',
    layers=3,
    initializer='amplitude',
    optimization_objective='cvar',
    cvar_alpha=0.5,
    optimizer='slsqp',
    maxiter=8,
    seed=342,
    restarts=2,
)
cvar_result = cvar_selector.optimize()
print({
    'selected_state': cvar_result.selected_key,
    'selected_gap': cvar_result.optimum_gap,
    'expected_gap': cvar_result.expected_gap,
    'optimum_probability': cvar_result.optimum_probability,
    'optimization_value': cvar_result.summary()['optimization_value'],
})


## 7. Reproducibility and scope

The command-line entry point scripts/run_benchmark.py evaluates six deterministic instances and writes JSON, CSV, PNG, and metadata artifacts. Running the notebook from the project directory reproduces the same model and API workflow interactively.

This package targets small instances that fit in an exact CPU state vector. Exact enumeration is a correctness reference with exponential scaling; the package does not claim a quantum speedup. Cloud execution, hardware noise, sampling error, and device calibration are outside this state-vector validation workflow.

## References

1. X. Jiang, Z. Chen, J. Zhang, Z. Yu, L. Wang, and H. Mei, QAOA-based MRMR Algorithm for Feature Selection, Proceedings of the 2023 International Conference on Advances in Artificial Intelligence and Applications, 2024.
2. OriginQ, pyqpanda-algorithm, source repository: https://github.com/OriginQ/pyqpanda-algorithm.
3. S. Hadfield et al., From the Quantum Approximate Optimization Algorithm to a Quantum Alternating Operator Ansatz, Algorithms, 2019.
4. A. Barkoutsos et al., Improving Variational Quantum Optimization using CVaR, Quantum, 2020.